# AIH Round 1 revision — Colab T4
Select Runtime → Change runtime type → T4 GPU. Upload `revision_colab_bundle.zip` when prompted. This trains ten grouped splits for three encoders and saves primary-seed checkpoints. Energy is GPU-board only; host and wall energy remain unmeasured. Run duration depends on the runtime; results are written incrementally.


In [ ]:
%pip install -q --upgrade regex "transformers>=4.40,<5" safetensors pandas numpy scipy scikit-learn


In [ ]:
from google.colab import files
from pathlib import Path
import zipfile, os
uploaded=files.upload()
name=next((n for n in uploaded if n.endswith(".zip")),None)
if name is None: raise ValueError("Upload revision_colab_bundle.zip")
root=Path("/content/aih_revision"); root.mkdir(exist_ok=True)
with zipfile.ZipFile(name) as archive:
    for member in archive.namelist():
        if not (root/member).resolve().is_relative_to(root.resolve()): raise ValueError("Unsafe archive path")
    archive.extractall(root)
os.chdir(root)
print("Ready:",root)


In [ ]:
import subprocess, sys
# Defaults: seeds 42,123,456,1001..1007; DistilBERT, PubMedBERT, BERT-tiny.
# For a quick first session use ["--seeds","42"] below, then resume remaining seeds.
extra=[]
with open("revision_colab_run.log","w") as log:
    process=subprocess.Popen([sys.executable,"-u","scripts/revision_colab.py"]+extra,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True)
    for line in process.stdout:
        print(line,end=""); log.write(line); log.flush()
    code=process.wait()
if code: raise RuntimeError(f"Revision run exited {code}; retain existing outputs and rerun only incomplete seeds")


Run the next cell to download both ZIP files and the console log. `revision_predictions.zip` contains predictions and board-energy surfaces; extract it at the local repository root. Give `revision_checkpoints.zip` to the Linux operator and extract at that repository root too. Checkpoints exceed GitHub normal-file limits: transfer the archive separately; do not commit weights. Colab local storage is temporary—download archives before disconnecting. If the browser blocks downloads, use the Files pane to download them from `/content/aih_revision/`.


In [ ]:
from google.colab import files
from pathlib import Path
for name in ("revision_predictions.zip", "revision_checkpoints.zip", "revision_colab_run.log"):
    path = Path("/content/aih_revision") / name
    if not path.is_file():
        raise FileNotFoundError(f"Missing {path}; check the run output above")
    files.download(str(path))
